In [1]:
from pyspark.sql import functions as F

In [2]:
import time
from spark_session import get_spark
import sources

In [3]:
spark = get_spark("lazy")
orders, products = sources.orders(spark), sources.products(spark)

In [8]:
t = time.perf_counter()
sales = (
    orders.where("quantity > 0")
    .join(F.broadcast(sources.products(spark)), "product_id")
    .withColumn("revenue", F.col("quantity") * F.col("price"))
    .groupBy("category").agg(F.round(F.sum("revenue")).alias("revenue"))
)

print(f"building the plan: , {time.perf_counter() - t:.3f}s")
t = time.perf_counter()
rows = sales.collect()
print(f"running it: , {time.perf_counter() - t:.3f}s", len(rows), "categories")

building the plan: , 0.090s
running it: , 0.513s 4 categories


In [ ]:
sales.explain("extended")

In [11]:
clean = sources.orders(spark).where("quantity > 0").dropDuplicates()
for label in ["no cache, 1st", "no cache, 2nd"]:
    t = time.perf_counter(); n = clean.count()
    print(f"{label}: {time.perf_counter() - t:.3f}s", n)
clean.cache()
for label in ["no cache, 1st", "no cache, 2nd"]:
    t = time.perf_counter(); n = clean.count()
    print(f"{label}: {time.perf_counter() - t:.3f}s", n)
clean.unpersist()

no cache, 1st: 2.527s 278486
no cache, 2nd: 1.453s 278486
no cache, 1st: 2.334s 278486
no cache, 2nd: 0.189s 278486


DataFrame[order_id: bigint, order_ts: timestamp, customer_id: int, product_id: string, quantity: int, status: string, updated_at: timestamp]